In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

load_dotenv()

# --- CONFIGURATION ---
PINECONE_API_KEY = "pcsk_Kc21M_HUydh8PpiJzWbEewSrS15ZPLM6ZHAMSKfKmz2VrrAm5s42iAFNFMz6JjwbDCrW5"
INDEX_NAME = "groooh"  
PDF_PATH = "data.pdf"  

# Set the environment variable explicitly so LangChain picks it up globally
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

def process_and_store_pdf(pdf_path: str, index_name: str):
    # 1. Initialize 1024-dimension Hugging Face Embedding Model
    print("Initializing 1024-dimension embedding model...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5"
    )

    # 2. Extract text pages from the PDF
    print(f"Reading PDF: {pdf_path}...")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # 3. Chunk text into paragraphs
    print("Splitting text into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=200, 
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)

    # 4. Generate embeddings and upsert straight to Pinecone
    print(f"Generating embeddings and uploading {len(chunks)} chunks to Pinecone index '{index_name}'...")
    
    vector_store = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=index_name
    )
    
    print("Success! Your PDF embeddings are stored in your Pinecone database.")

if __name__ == "__main__":
    process_and_store_pdf(PDF_PATH, INDEX_NAME)

C:\Users\Atif Ali\AppData\Local\Temp\ipykernel_15664\2975112406.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\groooh\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing 1024-dimension embedding model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1052.59it/s]


Reading PDF: data.pdf...
Splitting text into chunks...
Generating embeddings and uploading 159 chunks to Pinecone index 'groooh'...
Success! Your PDF embeddings are stored in your Pinecone database.
